# exp-raw-canonical-audit-01 — 원문 mutation / canonical profile 감사

목적은 H0의 `gene × functional event type` 압축이 버린 원문·정확 event 정보를 train-only로 확인하는 것입니다. 모델 학습·제출 생성은 하지 않습니다.

- train만 읽음, test 미열람
- WT·공백·NaN은 event 0개
- 모든 raw segment는 canonical event 또는 `OTHER`로 보존
- 고정 암종·유전자·mutation 규칙 없음

In [ ]:
from pathlib import Path
import json, subprocess, sys
from tqdm.auto import tqdm

ROOT = Path('/Users/admin/Documents/FinalProject/OZ_fianl_hackaton')
BASE = ROOT / 'experiments' / 'gs' / 'notebooks' / 'exp_model_011'
RUNNER = BASE / 'common' / 'run_raw_canonical_audit.py'
RESULT = BASE / 'result'
RUN_ID = 'exp-raw-canonical-audit-01'
RUN_EXPERIMENT = False  # 전체 train-only 감사를 실행할 때만 True
assert (ROOT / 'data' / 'raw' / 'train.csv').exists()
assert RUNNER.exists()
print({'runner': RUNNER, 'result_dir': RESULT, 'test_read': False})

In [ ]:
if RUN_EXPERIMENT:
    process = subprocess.Popen([sys.executable, str(RUNNER), '--run-id', RUN_ID], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in tqdm(process.stdout, desc='raw canonical audit', unit='line'):
        print(line, end='')
        tail = (tail + [line])[-80:]
    if process.wait() != 0:
        raise RuntimeError('audit runner failed:\n' + ''.join(tail))
else:
    print('RUN_EXPERIMENT=False: 기존 결과만 읽습니다.')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

summary = pd.read_csv(RESULT / f'{RUN_ID}_summary.csv')
transition = pd.read_csv(RESULT / f'{RUN_ID}_transition_summary.csv')
event_counts = pd.read_csv(RESULT / f'{RUN_ID}_event_type_counts.csv')
audit = json.loads((RESULT / f'{RUN_ID}_audit.json').read_text())
assert audit['test_read'] is False
assert audit['leakage_check'] is True
assert audit['nan_as_mutation_count'] == 0
assert audit['segment_conservation'] is True
display(summary)
display(transition)
display(event_counts)
audit

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
summary.set_index('profile_kind')['weighted_purity'].plot.bar(ax=axes[0], ylim=(0, 1), title='Weighted profile purity')
transition.set_index('transition')['merged_rows'].plot.bar(ax=axes[1], title='Rows merged by representation')
for ax in axes: ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

if audit['verdict'] == 'raw_token_model_candidate':
    print('판정: raw-token OOF 오류 다양성 감사 후보. 아직 성능 채택이나 제출 후보는 아닙니다.')
else:
    print('판정: canonical 압축에서 추가 정보가 확인되지 않아 raw-token 축을 종료합니다.')